In [101]:
import pandas as pd
import chess
import numpy as np

In [102]:
# test.pgn is only 1 game.
df = pd.read_csv("../tests/test.pgn", names=["data"], sep="|") # here we use sep as | because its not used anywhere in our data and we want only 1 col
df = df[
    (~df['data'].str.startswith('[')) # this tilde is negation
    | (df['data'].str.startswith('[Result'))
    | (df['data'].str.startswith('[PlyCou'))
    | (df['data'].str.startswith('[WhiteE'))
    | (df['data'].str.startswith('[BlackE'))
].reset_index(drop=True) # fix indices after column removal
df

,data
0,"[Result ""1/2-1/2""]"
1,"[PlyCount ""214""]"
2,"[WhiteElo ""3189""]"
3,"[BlackElo ""3125""]"
4,1. d4 d5 2. c4 e6 3. Nc3 c6 4. e3 Nf6 5. Nf3 c...
5,Bxc5 9. Qxd8+ Kxd8 10. e4 a6 11. Bxc4 b5 12. B...
6,Bf4 Rhc8 16. h3 h6 17. Bg3 Na5 18. Rac1 Rab8 1...
7,22. Rd6 Nb6 23. Rd2 Rc5 24. a3 Rbc8 25. Bh4+ f...
8,Rxc3 Rxc3 29. Bd6+ Ke8 30. Bxb4 Rxb3 31. Bd3 N...
9,Rxd4 a5 35. Bf8 e5 36. Rg4 Bc6 37. Bh7 Kd8 38....


In [103]:
df_work = df.copy()

df_work['data_c'] = df_work['data']
df_work['meta'] = df_work['data'].str.startswith('[')
df_work['data'] = df_work['data'].where(df_work['meta'] == True, '')
df_work['data_c'] = df_work['data_c'].where(df_work['meta'] == False, '')
df_work['data_c'] = df_work['data_c'].apply(lambda x: ' '.join(filter(lambda x: not x.endswith('.'), x.split(' ')))) #looks complicated but its just removing all move numbers by removing things that end with .

df_work

,data,data_c,meta
0,"[Result ""1/2-1/2""]",,True
1,"[PlyCount ""214""]",,True
2,"[WhiteElo ""3189""]",,True
3,"[BlackElo ""3125""]",,True
4,,d4 d5 c4 e6 Nc3 c6 e3 Nf6 Nf3 c5 Be2 dxc4 O-O ...,False
5,,Bxc5 Qxd8+ Kxd8 e4 a6 Bxc4 b5 Be2 Bb7 e5 Nd7 R...,False
6,,Bf4 Rhc8 h3 h6 Bg3 Na5 Rac1 Rab8 Nd2 Ba7 b3 Nc...,False
7,,Rd6 Nb6 Rd2 Rc5 a3 Rbc8 Bh4+ f6 exf6+ gxf6 Bg3...,False
8,,Rxc3 Rxc3 Bd6+ Ke8 Bxb4 Rxb3 Bd3 Nd5 Bg6+ Kd7 ...,False
9,,Rxd4 a5 Bf8 e5 Rg4 Bc6 Bh7 Kd8 Bd6 Rc3 Bf5 Ne7...,False


In [104]:
results = []

for d1, d2, is_meta in df_work.values:
    if is_meta:
        results += [d1]
        tmp = False
    else:
        if not tmp:
            results += [d2]
            tmp = True
        else:
            results[-1] = ' '.join([results[-1], d2])

print(results)

['[Result "1/2-1/2"]', '[PlyCount "214"]', '[WhiteElo "3189"]', '[BlackElo "3125"]', 'd4 d5 c4 e6 Nc3 c6 e3 Nf6 Nf3 c5 Be2 dxc4 O-O Nc6 dxc5 Bxc5 Qxd8+ Kxd8 e4 a6 Bxc4 b5 Be2 Bb7 e5 Nd7 Rd1 Ke7 Bf4 Rhc8 h3 h6 Bg3 Na5 Rac1 Rab8 Nd2 Ba7 b3 Nc6 Nf3 Nb4 Rd6 Nb6 Rd2 Rc5 a3 Rbc8 Bh4+ f6 exf6+ gxf6 Bg3 Rxc3 Rxc3 Rxc3 Bd6+ Ke8 Bxb4 Rxb3 Bd3 Nd5 Bg6+ Kd7 Nd4 Bxd4 Rxd4 a5 Bf8 e5 Rg4 Bc6 Bh7 Kd8 Bd6 Rc3 Bf5 Ne7 Be6 Rd3 Bc5 Bd7 Bxd7 Rxd7 Bb6+ Ke8 Bxa5 Nc6 Bb6 Rd3 Rg8+ Kf7 Ra8 h5 Kf1 Rd1+ Ke2 Ra1 Rc8 Ra2+ Ke1 Ne7 Rc5 Ra1+ Kd2 Ke6 Rxb5 Ra2+ Ke1 Rxa3 Ra5 Rb3 Be3 Nd5 Bc1 Nb4 Kd2 Nc6 Ra6 Kd5 Ra8 Rb4 Rf8 Ke6 Rh8 h4 Kc3 Kd5 Be3 Rc4+ Kd3 Nb4+ Ke2 Ke6 Re8+ Kf5 Rd8 Rc2+ Kf1 Nc6 Rh8 Rc4 Rh5+ Kg6 Rh6+ Kg7 Rh5 Nb4 Ke2 Nd5 Kf3 Kg6 Rh8 Kf7 Ra8 Rb4 Bc5 Rf4+ Ke2 Rc4 Be3 Rb4 Ra7+ Ke6 Bd2 Rb2 Kd1 Rb1+ Bc1 Rb8 Ke2 Nc3+ Kd3 Nb5 Ra5 Rd8+ Ke3 Nd4 Bb2 Nc2+ Ke2 Nb4 Bc3 Nd5 Bd2 Rg8 Ra6+ Kf5 Kf1 Rb8 Ra7 Rb1+ Ke2 Rb2 Ke1 Nf4 Bxf4 Kxf4 Rh7 Kg5 Kf1 f5 Rg7+ Kf6 Rg8 Rb5 Rf8+ Ke6 Rh8 Rb1+ Ke2 Rb2+ Kf3 Rb4 1/2-1/2', '[Result "1/2-

In [105]:
df_clean = pd.DataFrame(results, columns=['data'])
df_clean['game'] = df_clean['data'].where(~df_clean['data'].str.startswith('['), '')
df_clean['data'] = df_clean['data'].str.strip('[]')
df_clean['result'] = df_clean['data'].where(df_clean['data'].str.startswith('Result'), '')
df_clean['len'] = df_clean['data'].where(df_clean['data'].str.startswith('PlyCount'), '')
df_clean['welo'] = df_clean['data'].where(df_clean['data'].str.startswith('WhiteElo'), '')
df_clean['belo'] = df_clean['data'].where(df_clean['data'].str.startswith('BlackElo'), '')

df_clean = df_clean.drop('data', axis='columns')
df_clean['result'] = df_clean['result'].apply(lambda x: x.split(' ')[-1].strip('"'))
df_clean['len'] = df_clean['len'].apply(lambda x: x.split(' ')[-1].strip('"'))
df_clean['welo'] = df_clean['welo'].apply(lambda x: x.split(' ')[-1].strip('"'))
df_clean['belo'] = df_clean['belo'].apply(lambda x: x.split(' ')[-1].strip('"'))

df_clean

,game,result,len,welo,belo
0,,1/2-1/2,,,
1,,,214,,
2,,,,3189,
3,,,,,3125
4,d4 d5 c4 e6 Nc3 c6 e3 Nf6 Nf3 c5 Be2 dxc4 O-O ...,,,,
5,,1/2-1/2,,,
6,,,127,,
7,,,,3584,
8,,,,,3449
9,d4 Nf6 c4 c5 Nf3 cxd4 Nxd4 e6 Nc3 Bb4 Bd2 O-O ...,,,,


In [106]:
example = {'result': '', 'len': '', 'welo': '', 'belo': '', 'game': ''}
games = []

for game, result, len, welo, belo in df_clean.values:
    if result != '':
        example['result'] = result
    if len != '':
        example['len'] = len
    if welo != '':
        example['welo'] = welo
    if belo != '':
        example['belo'] = belo
    elif game != '':
        example['game'] = game
        games += [example]
        example = {'result': '', 'len': '', 'welo': '', 'belo': '', 'game': ''}

games = pd.DataFrame(games)
games

,result,len,welo,belo,game
0,1/2-1/2,214,3189,3125,d4 d5 c4 e6 Nc3 c6 e3 Nf6 Nf3 c5 Be2 dxc4 O-O ...
1,1/2-1/2,127,3584,3449,d4 Nf6 c4 c5 Nf3 cxd4 Nxd4 e6 Nc3 Bb4 Bd2 O-O ...


In [107]:
print("Data quality")

print(f"Empty result: {(games['result'] == '').sum()}")
print(f"Empty len:    {(games['len'] == '').sum()}")
print(f"Empty welo:   {(games['welo'] == '').sum()}")
print(f"Empty belo:   {(games['belo'] == '').sum()}")
print(f"Empty game:   {(games['game'] == '').sum()}")

print()
display(games.dtypes)

Data quality
Empty result: 0
Empty len:    0
Empty welo:   0
Empty belo:   0
Empty game:   0



result    object
len       object
welo      object
belo      object
game      object
dtype: object

In [108]:
games['result'] = games['result'].apply(lambda x: {'1-0': 'W', '1/2-1/2': 'D', '0-1': 'B'}[x])
games = games[games['welo'] != '']
games = games[games['belo'] != '']
games = games.astype({'len': np.int32, 'welo': np.int32, 'belo': np.int32})
games = games.sample(frac=1).reset_index(drop=True)

games

,result,len,welo,belo,game
0,D,214,3189,3125,d4 d5 c4 e6 Nc3 c6 e3 Nf6 Nf3 c5 Be2 dxc4 O-O ...
1,D,127,3584,3449,d4 Nf6 c4 c5 Nf3 cxd4 Nxd4 e6 Nc3 Bb4 Bd2 O-O ...


In [109]:
games_filt = games[np.minimum(games['welo'], games['belo']) >= 3000].reset_index(drop=True)
games_filt

,result,len,welo,belo,game
0,D,214,3189,3125,d4 d5 c4 e6 Nc3 c6 e3 Nf6 Nf3 c5 Be2 dxc4 O-O ...
1,D,127,3584,3449,d4 Nf6 c4 c5 Nf3 cxd4 Nxd4 e6 Nc3 Bb4 Bd2 O-O ...


In [110]:
print("Results rates:")
display(games['result'].value_counts())
games.describe().T

Results rates:


result
D    2
Name: count, dtype: int64

,count,mean,std,min,25%,50%,75%,max
len,2.0,170.5,61.518290,127.0,148.75,170.5,192.25,214.0
welo,2.0,3386.5,279.307179,3189.0,3287.75,3386.5,3485.25,3584.0
belo,2.0,3287.0,229.102597,3125.0,3206.00,3287.0,3368.00,3449.0


In [111]:
new_games = []

for game in games_filt['game']:
    moves = game.split(' ')[:-1]

    formatted_moves = []

    for i, move in enumerate(moves):
        if i % 2 == 0:
            player = 'W'
        else:
            player = 'B'

        move_number = (i + 2) // 2

        formatted_move = f"{player}{move_number}.{move}"
        formatted_moves.append(formatted_move)

    new_game = ' '.join(formatted_moves)
    new_games.append(new_game)

games_filt['game'] = new_games
games_filt

,result,len,welo,belo,game
0,D,214,3189,3125,W1.d4 B1.d5 W2.c4 B2.e6 W3.Nc3 B3.c6 W4.e3 B4....
1,D,127,3584,3449,W1.d4 B1.Nf6 W2.c4 B2.c5 W3.Nf3 B3.cxd4 W4.Nxd...
